# Clase 7: Redes Neuronales Convolucionales (CNN)

## Ejemplos prácticos para la clase

---
## 1. ¿Por qué NO usar un MLP para imágenes?

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# El problema de parámetros de un MLP con imágenes

resoluciones = {
    "MNIST (28x28x1)": 28 * 28 * 1,
    "CIFAR-10 (32x32x3)": 32 * 32 * 3,
    "ImageNet (224x224x3)": 224 * 224 * 3,
    "Full HD (1920x1080x3)": 1920 * 1080 * 3,
}

n_hidden = 1000  # neuronas en la primera capa oculta

print("=" * 65)
print(f"Parámetros de un MLP con {n_hidden} neuronas en la primera capa")
print("=" * 65)
print(f"{'Imagen':<25} {'Pixeles':>10} {'Parámetros':>15}")
print("-" * 55)
for nombre, pixeles in resoluciones.items():
    params = pixeles * n_hidden
    print(f"{nombre:<25} {pixeles:>10,} {params:>15,}")

print(f"\nPara ImageNet: ¡150 MILLONES de parámetros solo en la primera capa!")
print(f"Esto causa: overfitting masivo, lentitud, y uso excesivo de memoria.")
print(f"\nAdemás, el MLP APLANA la imagen → pierde toda la estructura espacial.")

---
## 2. La operación de convolución

In [ ]:
# Convolución manual: un filtro se desliza sobre la imagen

# Imagen 5x5
imagen = np.array([
    [1, 0, 1, 0, 1],
    [0, 1, 0, 1, 0],
    [1, 0, 1, 0, 1],
    [0, 1, 0, 1, 0],
    [1, 0, 1, 0, 1]
])

# Filtro 3x3 (detecta un patrón específico)
filtro = np.array([
    [1, 0, 1],
    [0, 1, 0],
    [1, 0, 1]
])

# Convolución manual
output_size = imagen.shape[0] - filtro.shape[0] + 1  # 5 - 3 + 1 = 3
output = np.zeros((output_size, output_size))

print("Convolución paso a paso:")
for i in range(output_size):
    for j in range(output_size):
        region = imagen[i:i+3, j:j+3]
        output[i, j] = np.sum(region * filtro)
        if i == 0 and j == 0:
            print(f"\nPosición ({i},{j}):")
            print(f"  Región: {region.flatten().tolist()}")
            print(f"  Filtro: {filtro.flatten().tolist()}")
            print(f"  Producto: {(region * filtro).flatten().tolist()}")
            print(f"  Suma: {output[i,j]:.0f}")

print(f"\nResultado (feature map):")
print(output)
print(f"\nDonde el filtro coincide con el patrón → valor alto (5)")
print(f"Donde no coincide → valor bajo (0)")

In [ ]:
# Visualizar la convolución

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(imagen, cmap='gray_r', vmin=0, vmax=1)
axes[0].set_title('Imagen (5x5)', fontsize=13)
for i in range(5):
    for j in range(5):
        axes[0].text(j, i, str(imagen[i, j]), ha='center', va='center', fontsize=12)

axes[1].imshow(filtro, cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('Filtro (3x3)', fontsize=13)
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, str(filtro[i, j]), ha='center', va='center', fontsize=12)

im = axes[2].imshow(output, cmap='YlOrRd', vmin=0, vmax=5)
axes[2].set_title('Feature Map (3x3)', fontsize=13)
for i in range(3):
    for j in range(3):
        axes[2].text(j, i, f'{output[i,j]:.0f}', ha='center', va='center', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=axes[2])

plt.suptitle('Operación de Convolución', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# Filtros para detección de bordes (horizontales, verticales, diagonales)

import torchvision
import torchvision.transforms as transforms

# Cargar una imagen real de MNIST
mnist = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
img, label = mnist[3]  # un dígito
img_np = img.squeeze().numpy()

# Filtros de detección de bordes
filtros = {
    'Bordes horizontales': np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype=np.float32),
    'Bordes verticales': np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=np.float32),
    'Bordes diagonales': np.array([[0, -1, -1], [1, 0, -1], [1, 1, 0]], dtype=np.float32),
    'Sharpen': np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32),
}

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

axes[0].imshow(img_np, cmap='gray')
axes[0].set_title(f'Original (dígito {label})', fontsize=12)
axes[0].axis('off')

for ax, (nombre, filtro) in zip(axes[1:], filtros.items()):
    # Aplicar convolución con PyTorch
    conv = nn.Conv2d(1, 1, 3, bias=False, padding=1)
    conv.weight.data = torch.tensor(filtro).unsqueeze(0).unsqueeze(0)
    with torch.no_grad():
        result = conv(img.unsqueeze(0)).squeeze().numpy()
    ax.imshow(result, cmap='gray')
    ax.set_title(nombre, fontsize=12)
    ax.axis('off')

plt.suptitle('Filtros de convolución aplicados a un dígito MNIST', fontsize=14)
plt.tight_layout()
plt.show()

print("Las CNN aprenden estos filtros AUTOMÁTICAMENTE durante el entrenamiento.")
print("Las primeras capas aprenden bordes, las profundas aprenden formas complejas.")

---
## 3. Conv2d en PyTorch

In [ ]:
# Entender las dimensiones de Conv2d

# Imagen RGB de 32x32
x = torch.randn(1, 3, 32, 32)  # (batch=1, canales=3, alto=32, ancho=32)
print(f"Input: {x.shape}  →  1 imagen, 3 canales (RGB), 32x32 píxeles")

# Convolución: 3 canales entrada → 16 filtros, kernel 3x3
conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
out1 = conv1(x)
print(f"\nConv2d(3→16, 3x3, padding=1): {out1.shape}")
print(f"  → 16 feature maps de 32x32 (mismo tamaño por padding=1)")

# Max Pooling
pool = nn.MaxPool2d(kernel_size=2, stride=2)
out2 = pool(out1)
print(f"\nMaxPool2d(2x2): {out2.shape}")
print(f"  → Tamaño se reduce a la mitad: 32→16")

# Segunda convolución
conv2 = nn.Conv2d(16, 32, 3, padding=1)
out3 = conv2(out2)
out4 = pool(out3)
print(f"\nConv2d(16→32, 3x3) + Pool: {out4.shape}")
print(f"  → 32 feature maps de 8x8")

# Parámetros
print(f"\nParámetros por capa:")
print(f"  Conv1: {sum(p.numel() for p in conv1.parameters()):,} (vs {3*32*32*16:,} para un MLP)")
print(f"  Conv2: {sum(p.numel() for p in conv2.parameters()):,}")
print(f"\n¡Las CNN son MUCHÍSIMO más eficientes que los MLP para imágenes!")

---
## 4. Max Pooling

In [ ]:
# Visualización de Max Pooling

feature_map = np.array([
    [1, 3, 2, 1],
    [0, 2, 1, 0],
    [6, 4, 3, 8],
    [1, 2, 5, 2]
], dtype=np.float32)

# Max Pooling 2x2
pooled = np.array([
    [max(feature_map[0,0], feature_map[0,1], feature_map[1,0], feature_map[1,1]),
     max(feature_map[0,2], feature_map[0,3], feature_map[1,2], feature_map[1,3])],
    [max(feature_map[2,0], feature_map[2,1], feature_map[3,0], feature_map[3,1]),
     max(feature_map[2,2], feature_map[2,3], feature_map[3,2], feature_map[3,3])]
])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Feature map original
axes[0].imshow(feature_map, cmap='YlOrRd', vmin=0, vmax=8)
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, f'{feature_map[i,j]:.0f}', ha='center', va='center', fontsize=14)
# Dibujar regiones 2x2
axes[0].axhline(y=1.5, color='black', linewidth=2)
axes[0].axvline(x=1.5, color='black', linewidth=2)
axes[0].set_title('Feature Map (4x4)', fontsize=13)

# Después de pooling
axes[1].imshow(pooled, cmap='YlOrRd', vmin=0, vmax=8)
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f'{pooled[i,j]:.0f}', ha='center', va='center',
                     fontsize=18, fontweight='bold')
axes[1].set_title('Max Pooling 2x2 (2x2)', fontsize=13)

plt.suptitle('Max Pooling: de cada región 2x2, quedarse con el máximo', fontsize=14)
plt.tight_layout()
plt.show()

print(f"max(1,3,0,2)=3  max(2,1,1,0)=2")
print(f"max(6,4,1,2)=6  max(3,8,5,2)=8")
print(f"\nReduce el tamaño a la mitad → menos parámetros → más robusto")

---
## 5. CNN completa: clasificar CIFAR-10

In [ ]:
# Cargar CIFAR-10 (10 clases de imágenes 32x32 color)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(testset, batch_size=64)

clases = ('avion', 'auto', 'pajaro', 'gato', 'ciervo',
          'perro', 'rana', 'caballo', 'barco', 'camion')

print(f"CIFAR-10: {len(trainset)} train, {len(testset)} test")
print(f"Clases: {clases}")

# Mostrar ejemplos
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    img, label = trainset[i]
    img_show = img.permute(1, 2, 0).numpy() * 0.5 + 0.5  # desnormalizar
    ax.imshow(img_show)
    ax.set_title(clases[label], fontsize=10)
    ax.axis('off')
plt.suptitle('Muestras de CIFAR-10 (32x32 color)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Definir la CNN

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Bloque 1: 3→32 canales, 32x32→16x16
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Bloque 2: 32→64 canales, 16x16→8x8
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Bloque 3: 64→128 canales, 8x8→4x4
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CNN()
total_params = sum(p.numel() for p in model.parameters())
print(f"Parámetros totales: {total_params:,}")

# Verificar dimensiones
dummy = torch.randn(1, 3, 32, 32)
output = model(dummy)
print(f"Input:  {dummy.shape}")
print(f"Output: {output.shape} (10 clases)")

In [ ]:
# Entrenar la CNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Entrenando en: {device}")

model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10
train_losses = []
train_accs = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    train_loss = running_loss / len(train_loader)
    train_acc = 100. * correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    print(f"Epoch {epoch+1:>2d}/{num_epochs} - Loss: {train_loss:.4f} - Train Acc: {train_acc:.1f}%")

In [ ]:
# Evaluar en test

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

test_acc = 100. * correct / total
print(f"\nAccuracy en test: {test_acc:.1f}%")

# Curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, num_epochs+1), train_losses, 'o-', color='steelblue', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, num_epochs+1), train_accs, 'o-', color='#2ecc71', linewidth=2)
axes[1].axhline(y=test_acc, color='#e74c3c', linestyle='--', label=f'Test: {test_acc:.1f}%')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training Accuracy')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Entrenamiento de CNN en CIFAR-10', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Ver predicciones

fig, axes = plt.subplots(3, 8, figsize=(16, 6))
model.eval()
with torch.no_grad():
    for i, ax in enumerate(axes.flat):
        img, label = testset[i]
        pred = model(img.unsqueeze(0).to(device)).argmax(1).item()
        color = 'green' if pred == label else 'red'
        img_show = img.permute(1, 2, 0).numpy() * 0.5 + 0.5
        ax.imshow(img_show)
        ax.set_title(f'{clases[pred]}', color=color, fontsize=10, fontweight='bold')
        ax.axis('off')

plt.suptitle('Predicciones de la CNN (verde=correcto, rojo=incorrecto)', fontsize=14)
plt.tight_layout()
plt.show()

---
## 6. Arquitecturas famosas: línea temporal

In [ ]:
# Línea temporal de arquitecturas

arquitecturas = [
    ("LeNet-5", 1998, 0.06, 99.0, "Dígitos (MNIST)"),
    ("AlexNet", 2012, 60, 84.0, "ImageNet - Inicio del deep learning"),
    ("VGGNet", 2014, 138, 92.0, "Solo filtros 3x3, profundidad"),
    ("ResNet-50", 2015, 25, 96.4, "Skip connections, +152 capas"),
    ("EfficientNet-B0", 2019, 5.3, 97.0, "Escalar inteligentemente"),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Parámetros por año
anios = [a[1] for a in arquitecturas]
params = [a[2] for a in arquitecturas]
nombres = [a[0] for a in arquitecturas]

axes[0].bar(range(len(arquitecturas)), params, color=['#3498db', '#e74c3c', '#f39c12', '#2ecc71', '#9b59b6'])
axes[0].set_xticks(range(len(arquitecturas)))
axes[0].set_xticklabels([f"{n}\n({a})" for n, a in zip(nombres, anios)], fontsize=10)
axes[0].set_ylabel('Millones de parámetros')
axes[0].set_title('Parámetros por arquitectura')
axes[0].grid(True, alpha=0.3, axis='y')

# Top-1 accuracy
accs = [a[3] for a in arquitecturas]
axes[1].plot(range(len(arquitecturas)), accs, 'o-', color='steelblue', linewidth=2, markersize=10)
for i, (n, acc) in enumerate(zip(nombres, accs)):
    axes[1].annotate(f"{n}\n{acc}%", (i, acc), textcoords="offset points",
                     xytext=(0, 10), ha='center', fontsize=9)
axes[1].set_xticks(range(len(arquitecturas)))
axes[1].set_xticklabels(anios, fontsize=11)
axes[1].set_ylabel('Top-1 Accuracy (%)')
axes[1].set_title('Evolución de la accuracy')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Evolución de las arquitecturas CNN', fontsize=14)
plt.tight_layout()
plt.show()

print("Tendencia: modelos más nuevos → MEJOR performance con MENOS parámetros")
print("No se trata de hacer redes más grandes, sino más inteligentes.")

---
## Resumen

| Concepto | Clave |
|----------|-------|
| MLP para imágenes | Mala idea: demasiados params, pierde estructura |
| Convolución | Filtro pequeño se desliza detectando patrones |
| Compartición de pesos | Mismo filtro en toda la imagen → eficiente |
| Jerarquía de features | Bordes → Texturas → Formas → Objetos |
| Max Pooling | Reduce tamaño, mantiene info importante |
| BatchNorm | Estabiliza y acelera el entrenamiento |
| ResNet | Skip connections para redes muy profundas |